# Bagré | 05: Product agreement, exploratory stress signals and QGIS
    **Goal:** harmonize outputs onto a common coarse grid, compare ET products over crop-dominated cells, and investigate whether low ET relative to reference demand coincides with warmer vegetated surfaces. No field measurements are available; this is not a validated crop-water-stress index.

## 1. Mount Drive (no Earth Engine authentication required)
    Run only after Notebooks 01–04 have produced all files. CPU is sufficient.

In [ ]:
%pip -q install rasterio pyproj
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import rasterio
from rasterio.warp import reproject, Resampling
from rasterio.transform import from_origin
from pyproj import Transformer
from pathlib import Path
from google.colab import drive
drive.mount('/content/drive')
ROOT = Path('/content/drive/MyDrive/Bagre_ET_Water_Stress')
MONTHS = [f'2021-{m:02d}' for m in range(1,5)]
required = [ROOT/'data'/'worldcover_2021.tif', ROOT/'data'/'weather_monthly.csv']
required += [ROOT/'data'/f'{p}_{m}.tif' for m in MONTHS for p in ['lst','veg','mod16','wapor']]
missing = [str(p) for p in required if not p.exists()]
assert not missing, f'Missing prerequisites: {missing}'
for sub in ['outputs','figures','qgis']: (ROOT/sub).mkdir(exist_ok=True)

## 2. Harmonize spatial resolution
    The 500 m UTM grid matches the approximate coarsest ET resolution. We aggregate the finer data with area-weighted averaging and keep missing values as missing. Cropland fraction uses valid 10 m labels. Coverage fraction of each input is also checked; we exclude cells with less than 80% valid source coverage. The common grid is for comparison, not super-resolution.

In [ ]:
CRS = 'EPSG:32630'
scale = 500
t = Transformer.from_crs('EPSG:4326', CRS, always_xy=True)
west,south,east,north = -0.59,11.41,-0.47,11.52
xs, ys = t.transform([west,west,east,east],[south,north,south,north])
left, bottom = np.floor(min(xs)/scale)*scale, np.floor(min(ys)/scale)*scale
right, top = np.ceil(max(xs)/scale)*scale, np.ceil(max(ys)/scale)*scale
W,H = int((right-left)/scale),int((top-bottom)/scale)
TRANSFORM = from_origin(left,top,scale,scale)
def aggregate(path, band=1, crop=False):
    with rasterio.open(path) as src:
        a = src.read(band, masked=True).astype('float32').filled(np.nan)
        valid = np.isfinite(a)
        if crop: a = np.where(valid, (a==40).astype('float32'), np.nan)
        dest = np.full((H,W), np.nan, dtype='float32')
        cov = np.zeros((H,W), dtype='float32')
        common = dict(src_transform=src.transform, src_crs=src.crs,
                      dst_transform=TRANSFORM, dst_crs=CRS,
                      resampling=Resampling.average)
        reproject(a,dest,src_nodata=np.nan,dst_nodata=np.nan,**common)
        reproject(valid.astype('float32'),cov,dst_nodata=0,**common)
        dest[cov<0.8] = np.nan
        return dest
crop = aggregate(ROOT/'data'/'worldcover_2021.tif',crop=True)
crop_domain = crop>=0.5
print('500m cells:', H,W, 'crop-dominated cells:', int(crop_domain.sum()))
assert crop_domain.any(), 'No crop-dominated cells: inspect coverage and AOI.'
weather = pd.read_csv(ROOT/'data'/'weather_monthly.csv').set_index('month')

## 3. Compare product estimates on paired support
    Only cells with both ET products and crop fraction ≥ 50% enter the comparison. Bias is WaPOR minus MOD16; RMSE measures agreement, not error against truth. Pearson correlation describes co-variation, not accuracy. Repeat using 30% and 70% crop fraction to check sensitivity. Pixels and months are spatially and temporally dependent, so we report descriptive statistics only.

In [ ]:
def agreement(x,y,mask):
    keep = mask & np.isfinite(x) & np.isfinite(y)
    x,y = x[keep],y[keep]
    if not len(x): return dict(n=0,bias_mm=np.nan,mae_mm=np.nan,rmse_mm=np.nan,r=np.nan)
    d=y-x
    r=np.corrcoef(x,y)[0,1] if len(x)>1 and x.std()>0 and y.std()>0 else np.nan
    return dict(n=len(x),bias_mm=d.mean(),mae_mm=np.abs(d).mean(),
                rmse_mm=np.sqrt(np.mean(d*d)),r=r)
check=agreement(np.array([1.,2.]),np.array([2.,3.]),np.array([True,True]))
assert check['bias_mm']==1 and check['rmse_mm']==1
metrics, layers = [],{}
for month in MONTHS:
    x=aggregate(ROOT/'data'/f'mod16_{month}.tif')
    y=aggregate(ROOT/'data'/f'wapor_{month}.tif')
    lst=aggregate(ROOT/'data'/f'lst_{month}.tif')
    ndvi=aggregate(ROOT/'data'/f'veg_{month}.tif',1)
    ndmi=aggregate(ROOT/'data'/f'veg_{month}.tif',2)
    for threshold in [0.3,0.5,0.7]:
        metrics.append(dict(month=month,crop_fraction_threshold=threshold,
                            **agreement(x,y,crop>=threshold)))
    et0=float(weather.loc[month,'et0_mm'])
    assert et0>0
    layers[month]=dict(mod16=x,wapor=y,lst=lst,ndvi=ndvi,ndmi=ndmi,
                       mod16_et0=x/et0,wapor_et0=y/et0)
table=pd.DataFrame(metrics)
display(table)
table.to_csv(ROOT/'outputs'/'product_agreement.csv',index=False)

## 4. Exploratory water-stress screening
    We compare only actively green, crop-dominated cells (NDVI ≥ 0.35) with paired ET and LST data. Flag cells that have both products below the monthly 25th percentile of ET/ET0 and LST above the 75th percentile. This is a relative screening rule, not a physiological threshold or validated CWSI. Crop type, seasonality, soil background, cloud gaps and weather can cause similar patterns. Repeat for NDVI threshold 0.25 and 0.45 to assess sensitivity.

In [ ]:
def screen(v, ndvi_threshold):
    valid=(crop_domain & (v['ndvi']>=ndvi_threshold) &
           np.isfinite(v['mod16_et0']) & np.isfinite(v['wapor_et0']) &
           np.isfinite(v['lst']))
    if valid.sum()<10:
        return np.full(crop.shape,np.nan,dtype='float32'), int(valid.sum())
    xm=np.quantile(v['mod16_et0'][valid],.25)
    ym=np.quantile(v['wapor_et0'][valid],.25)
    lt=np.quantile(v['lst'][valid],.75)
    flag=(v['mod16_et0']<xm)&(v['wapor_et0']<ym)&(v['lst']>lt)
    return np.where(valid,flag.astype('float32'),np.nan), int(valid.sum())
summary=[]
for month,v in layers.items():
    for threshold in [.25,.35,.45]:
        flag,n=screen(v,threshold)
        summary.append({'month':month,'ndvi_threshold':threshold,'eligible_cells':n,
                        'flagged_cells':int(np.nansum(flag)) if n>=10 else None})
        if threshold==.35: v['screen']=flag
stress=pd.DataFrame(summary)
display(stress)
stress.to_csv(ROOT/'outputs'/'screen_sensitivity.csv',index=False)

## 5. Export georeferenced layers and figures
    Open the GeoTIFFs from outputs in QGIS. Use fixed color ranges across dates and add a scale bar, legend, CRS, date, source and caveat. Screening: 0 = eligible but not flagged, 1 = flagged, nodata = not assessed. This is the 500 m comparison grid, not a 10 m field map.

In [ ]:
def save_raster(name,a):
    path=ROOT/'outputs'/f'{name}.tif'
    with rasterio.open(path,'w',driver='GTiff',height=H,width=W,count=1,
                       dtype='float32',crs=CRS,transform=TRANSFORM,nodata=-9999,
                       compress='deflate') as dst:
        dst.write(np.where(np.isfinite(a),a,-9999).astype('float32'),1)
save_raster('cropland_fraction',crop)
for month,v in layers.items():
    for name,a in v.items(): save_raster(f'{name}_{month}',a)
fig,axes=plt.subplots(3,4,figsize=(15,10),constrained_layout=True)
for j,(month,v) in enumerate(layers.items()):
    for i,(key,title,cmap,lo,hi) in enumerate([
        ('wapor_et0','WaPOR ET/ET0','YlGnBu',0,1.5),
        ('lst','LST °C','inferno',20,50),
        ('screen','Relative screening flag','Oranges',0,1)]):
        p=axes[i,j].imshow(v[key],cmap=cmap,vmin=lo,vmax=hi)
        axes[i,j].set_title(f'{month} {title}')
        axes[i,j].axis('off')
        fig.colorbar(p,ax=axes[i,j],shrink=.65)
fig.savefig(ROOT/'figures'/'water_stress_screening.png',dpi=160)
plt.show()
print('Exported',len(list((ROOT/'outputs').glob('*.tif'))),'GeoTIFFs')

In [ ]:
display(table.round(3))

In [ ]:
for month, v in layers.items():
    print(f"\n{month}")

    for product in ['mod16', 'wapor']:
        a = v[product][crop_domain]
        a = a[np.isfinite(a)]

        print(
            product,
            f"Mean: {a.mean():.2f}",
            f"Median: {np.median(a):.2f}",
            f"Max: {a.max():.2f}",
            "mm/month"
        )

**Final interpretation questions:** Which product gives higher monthly ET? How sensitive is agreement to crop fraction? Do flag counts change greatly with NDVI threshold? What extra field observations would establish actual stress? Report negative or inconclusive results as honestly as positive ones. Never use the screen to prescribe irrigation.